## Irrigated Farmland Data

This is the script used to pull the data on total irrigated farmland in Maricopa county from the NASS website. It was written by University of Michigan Gemini.

In [ ]:
import pandas as pd
import requests

API_KEY = "5775BA86-D883-3968-8C24-BF9A33351F65"
URL = "https://quickstats.nass.usda.gov/api/api_GET/"

# The exact 6 Census years we want to pull
census_years = [1997, 2002, 2007, 2012, 2017, 2022]
all_records = []

print("Running precise targeted timeline extraction...")

for year in census_years:
    print(f" -> Fetching data for year {year}...")

    # Strategy 1: Look for the primary Ag Land label
    params = {
        "key": API_KEY,
        "source_desc": "CENSUS",
        "year": year,
        "short_desc": "AG LAND, IRRIGATED - ACRES",
        "state_name": "ARIZONA",
        "county_name": "MARICOPA",
        "format": "JSON",
    }

    response = requests.get(URL, params=params)

    # Strategy 2: If Strategy 1 fails or returns empty, try the fallback Irrigation label
    if response.status_code != 200 or "data" not in response.json():
        params["short_desc"] = "IRRIGATION - ACRES IRRIGATED"
        response = requests.get(URL, params=params)

    # Process the data if we found a match
    if response.status_code == 200 and "data" in response.json():
        raw_data = response.json()["data"]

        for row in raw_data:
            # Look for the global total record (skip individual farm-size breakdowns)
            domain = row.get("domaincat_desc", "").upper()
            if "SPECIFIED" in domain or domain == "":
                val_str = str(row.get("Value", "")).replace(",", "").strip()

                # Ignore privacy flags like (D)
                if val_str.isdigit():
                    all_records.append(
                        {
                            "year": int(year),
                            "county_name": "MARICOPA",
                            "short_desc": row.get("short_desc"),
                            "value": int(val_str),
                        }
                    )
                    break  # Found the true total for this year, move to next year

# Convert the collected timeline array directly into pandas
if all_records:
    final_df = pd.DataFrame(all_records)
    final_df = final_df.sort_values(by="year").reset_index(drop=True)

    print("\n--- SUCCESS! True Macro Acreage Timeline Extracted ---")
    print(final_df[["year", "county_name", "short_desc", "value"]])

    # Save cleanly to your CSV spreadsheet
    final_df[["year", "county_name", "value"]].to_csv(
        "maricopa_irrigation_timeline.csv", index=False
    )
    print("\nCorrected array exported to: maricopa_irrigation_timeline.csv")
else:
    print("\nNo matching records could be found.")